**1) Demonstrate the effectiveness of RAG to cite sources and to prevent hallucination in an LLM.**

Imports

In [1]:

import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForCausalLM


C:\Users\patel\anaconda3\envs\ragenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load documents

In [2]:

docs = []
for i in range(1, 6):
    with open(f"Document{i}.txt", "r", encoding="utf-8") as f:
        docs.append(f.read().strip())
print(f"Loaded {len(docs)} documents.\n")

Loaded 5 documents.



Build a TF–IDF vectorizer

In [3]:

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)


Define query

In [4]:
query = "What is the role of activation functions in neural networks?"
print(f"Query: {query}")
q_vec = vectorizer.transform([query])


Query: What is the role of activation functions in neural networks?


Retrieve top-2 via cosine similarity

In [5]:

cosines = (tfidf_matrix @ q_vec.T).toarray().ravel()
top_idxs = np.argsort(cosines)[::-1][:2]
print("Retrieved Documents (top 2):")
for rank, idx in enumerate(top_idxs, start=1):
    snippet = docs[idx][:200].replace("\n"," ") + ("…" if len(docs[idx])>200 else "")
    print(f"  {rank}. Doc{idx+1}: {snippet}")
print()


Retrieved Documents (top 2):
  1. Doc2: Activation functions play a crucial role in neural networks by introducing non-linearity into the model. Without them, the network would behave like a simple linear regression, regardless of how many …
  2. Doc5: Deep learning is a subset of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes i…



Build RAG prompt

In [7]:
# build the prompt dynamically from docs:
context = "\n".join(f"[Doc{idx+1}] {docs[idx]}" for idx in top_idxs)

rag_prompt = f"""You are a helpful assistant.
Use ONLY the facts in the context below—do NOT add anything else.
Answer in a single bullet point; end the bullet with its source tag.

Context:
{context}

Question: What is the role of activation functions in neural networks?
Answer:"""

print("RAG Prompt:\n")
print(rag_prompt)


RAG Prompt:

You are a helpful assistant.
Use ONLY the facts in the context below—do NOT add anything else.
Answer in a single bullet point; end the bullet with its source tag.

Context:
[Doc2] Activation functions play a crucial role in neural networks by introducing non-linearity into the model. Without them, the network would behave like a simple linear regression, regardless of how many layers it has. Common activation functions include ReLU (Rectified Linear Unit), which is efficient and commonly used in hidden layers, and sigmoid or tanh, often used in binary classification or when smoother outputs are needed. These functions allow the network to learn and represent complex patterns in the data.
[Doc5] Deep learning is a subset of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration from biological neuroscience and is centered around stacking artificial

Load GPT-2 for non-RAG generation

In [8]:

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
lm = AutoModelForCausalLM.from_pretrained("distilgpt2")


Produce a RAG and Non-RAG answer

In [15]:
# Generate a RAG‐grounded answer


# Tokenize and run beam search
inputs1 = tokenizer(
    rag_prompt,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)
out1 = lm.generate(
    **inputs1,
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    num_beams=5,
    early_stopping=True
)

full = tokenizer.decode(out1[0], skip_special_tokens=True)
raw = full.split("Answer:")[-1].strip()
raw = re.split(r'\bQuestion:.*', raw, flags=re.IGNORECASE)[0].strip()
rag_answer = raw

# Generate a Non-RAG answer
nr_prompt = f"Question: {query}\nAnswer:"
inputs2 = tokenizer(nr_prompt, return_tensors="pt")
out2 = lm.generate(
    **inputs2,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    top_p=0.9,
    temperature=0.8
)
non_rag = tokenizer.decode(out2[0], skip_special_tokens=True).split("Answer:")[-1].strip()


Print Answers

In [16]:

print("RAG-Grounded Answer:")
print(rag_answer, "\n")

print("Non-RAG Answer:")
print(non_rag)


RAG-Grounded Answer:
Activation functions play a crucial role in neural networks by introducing non-linearity into the model. Without them, the network would behave like a simple linear regression, regardless of how many layers it has. Common activation functions include ReLU (Rectified 

Non-RAG Answer:
We're talking about the activation function of activation functions and the activation function of activation functions. These are neural networks that can respond to any stimulus and can respond to any stimulus.
It's not a simple function but we're talking about activation functions. They're involved in many of these things.
Now, the more you do it, the more the neural network you get.
What does activation function have in mind for your brain?
It's a function that is very important for the brain


**2) Implement a neural network classifier for the loan data with Decision as the output attribute. Prepare the data as needed. Come up with your best performing model by changing the size and number of hidden layers and activation functions.**

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [18]:
# Load dataset
df = pd.read_excel("loan.xlsx")

# Convert target column to binary
df['Decision'] = df['Decision'].map({'accept': 1, 'reject': 0})

# Define features and label
X = df.drop(columns=['Decision'])
y = df['Decision']

# Separate numerical and categorical columns
numeric_cols = ['Age', 'Time_at_address', 'Time_employed', 'Time_bank', 'Home_Expn', 'Balance']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])


## Data Preprocessing
- Scaled numerical values using StandardScaler.
- Converted categorical variables using OneHotEncoding.


In [19]:
# 1) First split raw X/y
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

# 2) Fit the preprocessor on TRAIN only, then transform both
X_train = preprocessor.fit_transform(X_train_raw)
X_test  = preprocessor.transform(X_test_raw)

print(f"Shapes → X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Labels → y_train: {y_train.shape}, y_test: {y_test.shape}")


Shapes → X_train: (343, 35), X_test: (86, 35)
Labels → y_train: (343,), y_test: (86,)


## Model Architecture
- Input layer 
- Hidden layer 
- Output layer 

In [21]:
# Build the neural network
model = Sequential([
    Dense(32, activation='relu', input_dim=X_train.shape[1]),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

# Evaluate the model
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print("\n=== Classification Report on Test Set ===")
print(classification_report(y_test, y_pred))


Epoch 1/50


C:\Users\patel\anaconda3\envs\ragenv\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4924 - loss: 0.6910 - val_accuracy: 0.6628 - val_loss: 0.6443
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6537 - loss: 0.6361 - val_accuracy: 0.7558 - val_loss: 0.6053
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6999 - loss: 0.6081 - val_accuracy: 0.7907 - val_loss: 0.5698
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7336 - loss: 0.5729 - val_accuracy: 0.7791 - val_loss: 0.5405
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7658 - loss: 0.5243 - val_accuracy: 0.8140 - val_loss: 0.5167
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7852 - loss: 0.4928 - val_accuracy: 0.8140 - val_loss: 0.4982
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7756 - loss: 0.4923 - val_accuracy: 0.7907 - val_loss: 0.4833
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8246 - loss: 0.4339 - val_accuracy: 0.7907 - val_loss: 0.4797
Ep